# lib_eval — Test 평가 공통 함수 (초안: core 담당 2026-11-26, 검증: eval 담당)

`05_test`, `07_core16`, `06_collect`가 `%run -i ./lib_eval.ipynb`로 불러 씁니다. 개별 노트북에서 고치지 말고 여기만 고칩니다.

표시: `[미정]` 팀 결정 전, `[미정·기본값]` 일단 넣은 값, `[출처 확인 필요]` 문헌 담당 확인 대기.
이 노트북을 직접 열면 맨 아래 칸이 가짜 데이터로 함수들을 점검합니다.

In [ ]:
if "FEATURE_SPEC" not in globals():   # 이 노트북을 직접 열었을 때
    SKIP_LIB_TESTS = True
    %run -i ./00_config.ipynb
    %run -i ./lib_prep.ipynb
    SKIP_LIB_TESTS = False
import json, joblib, time, hashlib
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from xgboost import DMatrix
import matplotlib.pyplot as plt
from matplotlib import font_manager

def _lib_eval_hash():
    p = NB_DIR / "lib_eval.ipynb"
    if "notebook_code_hash" in globals():
        return notebook_code_hash(p)
    cells = json.loads(p.read_text(encoding="utf-8"))["cells"]
    src = "\n".join("".join(c["source"]) for c in cells if c["cell_type"] == "code")
    return hashlib.sha256(src.encode("utf-8")).hexdigest()[:12]

LIB_EVAL_HASH = _lib_eval_hash()
print(f"[lib_eval] 해시 {LIB_EVAL_HASH}")

def set_korean_font():
    names = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
        if name in names:
            plt.rcParams["font.family"] = name; break
    plt.rcParams["axes.unicode_minus"] = False

def tidy(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)

MODELS = {"logit": "① 로지스틱", "xgb": "② XGBoost", "xgb_mono": "③ 단조 XGBoost"}
OUTS = [out for _, (out, *_r) in FEATURE_SPEC.items()]
CAT_OUTS = [out for _, (out, tf, *_r) in FEATURE_SPEC.items() if tf == "cat"]
SIGN_OF = {out: s for _, (out, _, _, s, _) in FEATURE_SPEC.items() if s != 0}   # 제약 변수 20개 (부호표 2026-11-26 확정)

def source_of(col):
    """모형 입력 열 → 원천 변수 (원핫·결측 지시를 원천으로 묶음)"""
    if col in OUTS: return col
    if col.endswith("_na") and col[:-3] in OUTS: return col[:-3]
    for c in CAT_OUTS:
        if col.startswith(c + "_"): return c
    raise KeyError(col)

def predict(prep, model, X):
    return model.predict_proba(prep.transform(X))[:, 1]

def load_frozen_test_inputs(freeze_dir=None, test_dir=None):
    """본 Test 실행(05_test DRY_RUN=False)용 원스텝 게이트:
    1) verify_freeze()로 사전 고정본·코드·계약·데이터·모형·selection.json 일치 검증
    2) test_hold/test_prep_meta.json이 있으면 test_*.parquet sha256·고정본 연계 검증
    반환: dict(freeze=..., selection=..., selected_model=..., a_star=..., a_star_by_model=...)
    """
    info = verify_freeze(freeze_dir=freeze_dir, require_final=True)
    sel = info["selection"]
    assert sel and sel.get("valid") is True, "고정된 selection.json이 유효하지 않음"
    ref = sel.get("ref_model", REF_MODEL)
    a_map = sel.get("a_star") or {}
    assert ref in a_map, f"selection.json에 a_star[{ref!r}] 없음"

    tdir = Path(test_dir) if test_dir is not None else (ROOT / "test_hold")
    tp_meta_path = tdir / "test_prep_meta.json"
    assert tp_meta_path.exists(), f"test_prep_meta.json 없음({tp_meta_path}) → 00b_test_prep(DRY_RUN=False)를 먼저 실행하세요."
    tp = json.loads(tp_meta_path.read_text(encoding="utf-8"))
    assert tp.get("dry_run") is False, "test_prep_meta.json이 dry_run 결과임"
    assert tp.get("freeze_record_sha256") == info["freeze_record_sha256"], (
        "test_prep_meta.json이 현재 사전 고정본과 다른 고정 시점에 만들어짐"
    )
    tfiles = tp.get("test_files") or {}
    req_tf = ("test_X_raw.parquet", "test_meta.parquet")
    assert all(isinstance(tfiles.get(fn), str) and len(tfiles[fn]) == 64 for fn in req_tf), (
        f"test_prep_meta.json의 test_files에 {req_tf} sha256이 모두 있어야 합니다."
    )
    for fname in req_tf:
        assert sha256_file(tdir / fname) == tfiles[fname], f"test_hold/{fname} 해시 불일치"
    return dict(
        freeze=info,
        selection=sel,
        selected_model=sel["selected_model"],
        a_star=float(a_map[ref]),
        a_star_by_model=a_map,
    )

## 6-A 판별력·보정

In [ ]:
def ks_stat(y, p):
    return stats.ks_2samp(p[y == 1], p[y == 0]).statistic

def metrics_6a(y, P, ref=None):
    ref = REF_MODEL if ref is None else ref
    t = pd.DataFrame([dict(model=m, auc=roc_auc_score(y, p), ks=ks_stat(y, p),
                           brier=brier_score_loss(y, p), logloss=log_loss(y, p)) for m, p in P.items()]).set_index("model")
    if ref in t.index:
        for c in ("auc", "ks", "brier", "logloss"):
            # [출처 확인 필요] PoM 정의: Koklev(2025) 재인용. 지금은 '② 대비 상대 차이'로 계산함
            t[f"pom_{c}"] = (t[c] - t.loc[ref, c]) / abs(t.loc[ref, c])
    return t.reset_index()

## 6-B SR, 승인율별 곡선, paired bootstrap

In [ ]:
def n_approve(n, a):
    return max(int(round(a * n)), 2)          # model_template.sr_curve와 같은 규칙

def approve_mask(p, a):
    m = np.zeros(len(p), bool); m[np.argsort(p, kind="stable")[:n_approve(len(p), a)]] = True
    return m

def sr_point(p, e, a, w=None):
    """with_treasury SR: 승인 = 초과수익률, 거절 = 0. w가 있으면 금액가중(견고성용)."""
    r = np.where(approve_mask(p, a), e, 0.0)
    if w is None:
        sd = r.std(ddof=1)
        return np.nan if sd == 0 else r.mean() / sd
    w = np.asarray(w, float) / np.sum(w); m = (w * r).sum()
    v = (w * (r - m) ** 2).sum() * len(r) / (len(r) - 1)   # [미정·기본값] 금액가중 분산 = 가중 모분산 × n/(n−1)
    return np.nan if v <= 0 else m / np.sqrt(v)

def sr_by_rate(P, e, rates, w=None):
    return pd.DataFrame({m: [sr_point(p, e, a, w) for a in rates] for m, p in P.items()}, index=pd.Index(rates, name="rate"))

def _sr_counts(order, cnt, e_sorted, k, n):
    c = cnt[order]; before = np.cumsum(c) - c
    take = np.clip(k - before, 0, c)          # 그 대출의 복제본 중 승인되는 수 (경계 대출은 일부만)
    s1 = (take * e_sorted).sum(); s2 = (take * e_sorted * e_sorted).sum()
    mean = s1 / n
    var = max((s2 - n * mean * mean) / (n - 1), 0.0)
    return np.nan if var == 0.0 else mean / np.sqrt(var)

def paired_bootstrap_sr(P, e, a, n_boot, seed=None):
    """대출 단위 복원추출. 회차마다 그 재표본 안에서 승인율 a를 다시 적용하고 모든 모형의 SR을 같은 재표본으로 계산."""
    seed = SEED if seed is None else seed
    n = len(e); k = n_approve(n, a); rng = np.random.default_rng(seed)
    orders = {m: np.argsort(p, kind="stable") for m, p in P.items()}
    e_sorted = {m: e[o] for m, o in orders.items()}
    out = {m: np.empty(n_boot) for m in P}
    for i in range(n_boot):
        cnt = np.bincount(rng.integers(0, n, n), minlength=n)
        for m, o in orders.items():
            out[m][i] = _sr_counts(o, cnt, e_sorted[m], k, n)
    return pd.DataFrame(out)

def noninferiority_row(name, d_point, d_boot, delta, alpha=None, sr_ref=None):
    alpha = ALPHA if alpha is None else alpha
    q_hi = float(np.quantile(d_boot, 1 - alpha))
    if delta is None:
        verdict = "δ 미정"                                        # [미정] DELTA
    else:
        verdict = "비열등" if q_hi < delta else "비열등 아님"
    return dict(비교=name, dSR=d_point, ci_lo=float(np.quantile(d_boot, 0.025)), ci_hi=float(np.quantile(d_boot, 0.975)),
                q_975=q_hi, delta=delta, 판정=verdict, PoM=(d_point / abs(sr_ref) if sr_ref else np.nan))  # PoM [출처 확인 필요]

## 견고성 (같은 예측값으로 다시 계산만)

In [ ]:
def hpr_excess(meta):
    # [출처 확인 필요·확인 필요] HPR 초과수익 = 연율화 HPR − 같은 만기 국채. 01_prep의 hpr_ann 정의와 맞는지 core 담당 확인
    return (meta["hpr_ann"] - meta["rf_ann"]).to_numpy()

def robustness_table(P, meta, a, ref=None, others=("xgb_mono", "logit"), band=None):
    ref = REF_MODEL if ref is None else ref
    band = ROBUST["rate_band"] if band is None else band
    e = meta["excess_ret"].to_numpy(); w = meta["loan_amnt"].to_numpy(); eh = hpr_excess(meta)
    band_rates = RATES[(RATES >= band[0] - 1e-9) & (RATES <= band[1] + 1e-9)]
    rows = []
    for o in [m for m in others if m in P]:
        main = sr_point(P[ref], e, a) - sr_point(P[o], e, a)
        vals = {"금액가중": sr_point(P[ref], e, a, w) - sr_point(P[o], e, a, w),
                "HPR": sr_point(P[ref], eh, a) - sr_point(P[o], eh, a),
                f"승인율 {band[0]}~{band[1]} 평균": float(np.mean([sr_point(P[ref], e, r) - sr_point(P[o], e, r) for r in band_rates]))}
        # [미정·기본값] 승인율 구간은 '구간 평균 ΔSR'의 부호로 판정 (구간 안 모든 점의 부호로 볼지는 미정)
        for k, v in vals.items():
            rows.append(dict(비교=f"{MODELS[ref][:1]} − {MODELS[o][:1]}", 분석=k, 주분석_dSR=main, dSR=v,
                             부호일치=bool(np.sign(v) == np.sign(main))))
    t = pd.DataFrame(rows)
    t["견고(비교별)"] = t.groupby("비교")["부호일치"].transform("all")   # 규칙: ROBUST['rule']
    return t

## 6-C 단조성 위반율, 거절 사유, 안정성

In [ ]:
def violations(prep, model, X_rej, sd, signs=None, eps=None, tol=None):
    """거절된 차주의 제약 변수를 하나씩 '유리한 방향'(부호 +1이면 감소, −1이면 증가)으로 eps×SD 움직였을 때 p̂가 tol 넘게 오르면 위반."""
    signs = SIGN_OF if signs is None else signs
    eps = EXPLAIN["perturb_sd"] if eps is None else eps
    tol = EXPLAIN["viol_tol"] if tol is None else tol
    p0 = predict(prep, model, X_rej)
    cols = [c for c in signs if c in X_rej.columns]
    V = np.zeros((len(X_rej), len(cols)), bool); valid = np.zeros_like(V)
    for j, c in enumerate(cols):
        Xp = X_rej.copy()
        Xp[c] = Xp[c] - signs[c] * eps * sd[c]
        valid[:, j] = X_rej[c].notna().to_numpy()          # 결측은 움직이지 않음 → 판정에서 제외
        V[:, j] = (predict(prep, model, Xp) - p0 > tol) & valid[:, j]
    return dict(borrower_rate=float(V.any(1).mean()),      # 주 지표 [미정·기본값]: 위반이 하나라도 있는 차주 비율
                pair_rate=float(V.sum() / valid.sum()),     # 보조: (차주, 변수) 쌍 비율
                by_var=pd.Series(V.sum(0) / np.maximum(valid.sum(0), 1), index=cols))

def contributions(prep, model, X, ref_mean=None):
    """로그오즈 기여도를 원천 변수로 합산. XGB = TreeSHAP(pred_contribs), 로지스틱 = β_j(x_ij − x̄_j)."""
    Xt = prep.transform(X); cols = list(Xt.columns)
    if isinstance(model, LogisticRegression):
        C = (Xt.to_numpy() - ref_mean.to_numpy()) * model.coef_[0]   # x̄ = Dev 평균 [미정·기본값]
    else:
        C = model.get_booster().predict(DMatrix(Xt), pred_contribs=True)[:, :-1]
    C = pd.DataFrame(C, columns=cols, index=X.index)
    return C.T.groupby([source_of(c) for c in cols]).sum().T

def top_reasons(Cagg, k=None):
    """위험을 올리는 방향(기여 > 0)만, 큰 순서로 k개. 양수가 k개보다 적으면 있는 만큼."""
    k = EXPLAIN["top_k"] if k is None else k
    V = Cagg.to_numpy(); names = np.array(Cagg.columns)
    idx = np.argsort(-V, axis=1)[:, :k]
    return [[names[j] for j in row if V[i, j] > 0] for i, row in enumerate(idx)]

def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not (a | b) else len(a & b) / len(a | b)

def rbo_at_k(l1, l2, p=None, k=None):
    # [출처 확인 필요] Webber et al.(2010) RBO를 깊이 k에서 자르고 [0,1]로 정규화한 보조 지표
    p = EXPLAIN["rbo_p"] if p is None else p
    k = EXPLAIN["top_k"] if k is None else k
    s = sum(p ** (d - 1) * len(set(l1[:d]) & set(l2[:d])) / d for d in range(1, k + 1))
    return s * (1 - p) / (1 - p ** k)

def stability(model_files, X_obs, ref_X=None, n_pairs=None, seed=None):
    """부트스트랩 모형마다 같은 차주들의 Top-k를 구하고, 무작위 모형 쌍의 Jaccard·RBO를 차주 평균."""
    n_pairs = EXPLAIN["jaccard_pairs"] if n_pairs is None else n_pairs
    seed = SEED if seed is None else seed
    lists = []
    for f in model_files:
        prep, model = joblib.load(f)
        ref_mean = prep.transform(ref_X).mean() if isinstance(model, LogisticRegression) else None
        lists.append(top_reasons(contributions(prep, model, X_obs, ref_mean)))
    B_ = len(lists); allp = [(i, j) for i in range(B_) for j in range(i + 1, B_)]
    rng = np.random.default_rng(seed)
    pairs = allp if len(allp) <= n_pairs else [allp[i] for i in rng.choice(len(allp), n_pairs, replace=False)]
    jac = [np.mean([jaccard(a, b) for a, b in zip(lists[i], lists[j])]) for i, j in pairs]
    rbo = [np.mean([rbo_at_k(a, b) for a, b in zip(lists[i], lists[j])]) for i, j in pairs]
    return dict(n_models=B_, n_pairs=len(pairs), jaccard=float(np.mean(jac)), jaccard_sd=float(np.std(jac, ddof=1)) if len(jac) > 1 else np.nan,
                rbo=float(np.mean(rbo)))

def pdp_1d(prep, model, X, col, qs=np.linspace(0.05, 0.95, 19)):
    """원 단위 분위수 격자에서 평균 p̂ (부록 C, 부호 점검용)"""
    grid = np.unique(np.nanquantile(X[col], qs)); out = []
    for g in grid:
        Xg = X.copy(); Xg[col] = g; out.append(predict(prep, model, Xg).mean())
    return pd.Series(out, index=grid)

In [ ]:
# 점검 (이 노트북을 직접 열었을 때만): 가짜 데이터
if not globals().get("SKIP_LIB_TESTS", False):
    rng = np.random.default_rng(0); n = 2000
    p = rng.random(n); e = rng.normal(0.01, 0.1, n); a = 0.5
    # (1) 재표본 SR(복제본 수 방식) = 재표본을 실제로 만들어 계산한 SR
    idx = np.random.default_rng(1).integers(0, n, n); cnt = np.bincount(idx, minlength=n)
    o = np.argsort(p, kind="stable")
    assert np.isclose(_sr_counts(o, cnt, e[o], n_approve(n, a), n), sr_point(p[idx], e[idx], a))
    # (2) 같은 예측이면 ΔSR = 0, 모든 회차
    bs = paired_bootstrap_sr({"x": p, "y": p.copy()}, e, a, 50)
    assert np.allclose(bs["x"], bs["y"])
    # (3) 금액가중에 모두 같은 가중치 = 동일가중, 분산 0이면 NaN
    assert np.isclose(sr_point(p, e, a, np.ones(n)), sr_point(p, e, a))
    assert np.isnan(sr_point(p, np.zeros(n), a))
    # (4) Jaccard·RBO 경계값
    assert jaccard(["a", "b"], ["a", "b"]) == 1 and jaccard(["a"], ["b"]) == 0 and np.isclose(rbo_at_k(list("abc"), list("abc")), 1)
    print("[PASS] lib_eval 점검 5개")